In [0]:
schema = 'fifa_bi_dev.silver_schema'

In [0]:
seed_correct_name = spark.read.table(f'{schema}.seed_correct_name')
seed_correct_name.show()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

df = spark.read.table("fifa_bi_dev.bronze_schema.world_cup_matches")

japan_cities = [
    "Sapporo", "Ibaraki", "Rifu", "Saitama", "Shizuoka",
    "Oita", "Niigata", "Yokohama", "Osaka", "Kobe"
]

korea_cities = [
    "Jeju", "Suwon", "Ulsan", "Gwangju", "Seoul",
    "Incheon", "Daegu", "Busan", "Jeonju", "Daejeon"
]


result = df.withColumn('year', col('year').cast('int'))\
            .withColumn('tournament_name', concat_ws(' ', lit('FIFA World Cup'), col('year')))\
            .withColumn('date', to_date(concat_ws("-", col('year'), month(to_date(col('date'), 'dd-MMM-yy')), day(to_date(col('date'), 'dd-MMM-yy'))),'yyyy-M-d'))\
            .withColumn('month', date_format(col('date'), 'yyyy-MM'))\
            .withColumn('time', date_format(to_timestamp(trim(col('time')), 'HH:mm'),'HH:mm'))\
            .withColumn('homegoals', col('homegoals').cast('int'))\
            .withColumn('awaygoals', col('awaygoals').cast('int'))\
            .withColumn('city', trim(col('city')))\
            .withColumn('observation', trim(col('observation')))\
            .withColumn('host_country',when(col('city').isin(japan_cities), 'Japan')
                            .when(col('city').isin(korea_cities), 'South Korea')
                            .when(trim(col('country')) == 'England', 'United Kingdom')
                            .when(trim(col('country')) == 'USA', 'United States')
                            .otherwise(trim(col('country'))))


#fix currupted stadium name

result = result.alias('r').join(seed_correct_name.alias('s'), col('r.stadium') == col('s.currupted_name'), 'left')\
            .withColumn('stadium', when(col('actual_name').isNull(), col('stadium')).otherwise(col('actual_name')))\
                .drop('currupted_name', 'actual_name')


#fix currupted city name

result = result.alias('r').join(seed_correct_name.alias('s'), col('r.city') == col('s.currupted_name'), 'left')\
            .withColumn('city', when(col('actual_name').isNull(), col('city')).otherwise(col('actual_name')))\
                .drop('currupted_name', 'actual_name')


#fix currupted country name

result = result.alias('r').join(seed_correct_name.alias('s'), col('r.hometeam') == col('s.currupted_name'), 'left')\
            .withColumn('hometeam', when(col('actual_name').isNull(), col('hometeam')).otherwise(col('actual_name')))\
                .drop('currupted_name', 'actual_name')

result = result.alias('r').join(seed_correct_name.alias('s'), col('r.awayteam') == col('s.currupted_name'), 'left')\
            .withColumn('awayteam', when(col('actual_name').isNull(), col('awayteam')).otherwise(col('actual_name')))\
                .drop('currupted_name', 'actual_name')

result.show()


In [0]:
#drop dublicates (remove matches at same datetime, within same round and between same team)

w = Window.partitionBy('year','date', 'time', 'round', 'hometeam', 'awayteam').orderBy('date')

result = result.withColumn('rank', row_number().over(w)).filter(col('rank') == 1).drop('rank')

#define match_stage - Group Stage, 2nd Group Stage, Knockout Stage

result = result.withColumn('match_stage', 
                when((col('year') == 1982) & (col('date') >= lit('1982-06-28'))  & (col('round').like('Group%')), '2nd Group Stage')\
                .when((col('year') == 1950) & (col('round').like('Group 6%')), '2nd Group Stage')\
                .when((col('year').isin(1974, 1978)) & (col('round').isin('Group A', 'Group B')), '2nd Group Stage')\
                .when(col('round').like('Group%'), 'Group Stage')\
                .otherwise('Knockout Stage'))


#fixture_id & match_id
result = result.withColumn("team_1", least(col("hometeam"), col("awayteam")))\
                .withColumn("team_2", greatest(col("hometeam"), col("awayteam")))

fixture_window = Window.partitionBy('year').orderBy("year","match_stage","team_1","team_2")

result = result.withColumn("fixture_id",concat(lit("F"),col("year"),1000+dense_rank().over(fixture_window)))

match_window = Window.orderBy("year","date","time")

result = result.withColumn("match_id",concat(lit("M"),col("year"),10000+row_number().over(match_window)))


#mark replay  - (flag matches within same year, same match_stage and between same team)
ww = Window.partitionBy('year' ,'match_stage', 'team_1', 'team_2').orderBy('date')

result = result.withColumn('rank', row_number().over(ww))\
                .withColumn('replay_flag', when(col('rank') == 2, 'Replay').otherwise(lit(''))).drop('rank')



#match_finish_type & penalties -
result = result.withColumn('match_finish_type', when(col('observation').like('%extra time%'), lit('Extra Time'))
                           .when(col('observation').like('%penalties%'), lit('Penalties'))
                           .when(col('observation').like('%Golden Goal%'), lit('Golden Goal'))
                           )


result = result.withColumn('homepenalties',regexp_extract(col('observation'), r'\((\d+)\s*-\s*(\d+)\)', 1))\
            .withColumn('awaypenalties',regexp_extract(col('observation'), r'\((\d+)\s*-\s*(\d+)\)', 2))

result = result.withColumn("homepenalties",when(col("homepenalties") == "", None).otherwise(col("homepenalties")).cast("int"))\
    .withColumn("awaypenalties",when(col("awaypenalties") == "", None).otherwise(col("awaypenalties")).cast("int"))







#result.printSchema()

#result.display()
result.show()

In [0]:
#team1 and team2 goals and penalties -
matches = result.withColumn('team_1_goals', when(col('team_1') == col('hometeam'), col('homegoals')).otherwise(col('awaygoals')))\
            .withColumn('team_2_goals', when(col('team_2') == col('hometeam'), col('homegoals')).otherwise(col('awaygoals')))\
            .withColumn('team_1_penalties', when(col('team_1') == col('hometeam'), col('homepenalties')).otherwise(col('awaypenalties'))) \
            .withColumn('team_2_penalties', when(col('team_2') == col('hometeam'), col('homepenalties')).otherwise(col('awaypenalties'))) 
matches.show()


In [0]:
team = spark.read.table('fifa_bi_dev.silver_schema.seed_team_codes')

#team_country -


final = matches.alias('m1').join(team.alias('t'), col('m1.team_1') == col('t.team_name'))\
                .withColumn('team_1_code', col('team_code'))\
                .withColumn('team_1_country', col('country_name')).drop('team_name','team_code','country_name')

final = final.alias('m2').join(team.alias('t'), col('m2.team_2') == col('t.team_name'))\
                .withColumn('team_2_code', col('team_code'))\
                .withColumn('team_2_country', col('country_name')).drop('team_name','team_code','country_name')



#match_winner -

final = final.withColumn('winner_country', 
                           when((col('match_finish_type').isNull()) & (col('team_1_goals') > col('team_2_goals')), col('team_1_country'))
                           .when((col('match_finish_type').isNull()) & (col('team_1_goals') < col('team_2_goals')), col('team_2_country'))
                           .when((col('match_finish_type').isNull()) & (col('team_1_goals') == col('team_1_goals')), lit('--'))
                           .when((col('match_finish_type')=='Extra Time') & (col('team_1_goals') > col('team_2_goals')), col('team_1_country'))
                           .when((col('match_finish_type')=='Extra Time') & (col('team_1_goals') < col('team_2_goals')), col('team_2_country'))
                           .when((col('match_finish_type')=='Golden Goal') & (col('team_1_goals') > col('team_2_goals')), col('team_1_country'))
                           .when((col('match_finish_type')=='Golden Goal') & (col('team_1_goals') < col('team_2_goals')), col('team_2_country'))
                           .when((col('match_finish_type')=='Penalties') & (col('team_1_penalties') > col('team_2_penalties')), col('team_1_country'))
                           .when((col('match_finish_type')=='Penalties') & (col('team_1_penalties') < col('team_2_penalties')), col('team_2_country'))


                           )




final.display()

In [0]:
final.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f'{schema}.silver_world_cup_matches')
print(f'table_name: silver_world_cup_matches is updated')